Instalisasi Faker

In [53]:
!pip install faker

Import library

In [54]:
import numpy as np
import pandas as pd
from faker import Faker
import random

Membuat data sintetis seed = 7

In [55]:
SEED = 7
np.random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transef Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range (1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max",
                                                           "Basic", ""])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint (1, 5)

  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",","."),
      f"{harga_dasar}.0",
      f"{harga_dasar}",
  ]

  harga = random.choice(harga_variants)

  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants= [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d%m%Y"),
                 tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() <0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None])

  rows.append({
      "transaction_id": trx_id,
      "customer_name": nama_pelanggan,
      "product_name": produk.strip(),
      "category": kategori,
      "price": harga,
      "quantity": qty,
      "payment_method": metode,
      "transaction_date": tanggal,
      "shipping_city": kota,
      "rating": rating,
  })


df = pd.DataFrame(rows)

for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df_mentah = df.copy()
df.to_csv("transaksi_mentah.csv", index=False)
print ("Jumlah baris:", len(df))

Jumlah baris: 515


Pendeteksi dan penanganan missing value

In [56]:
print(df.isnull().sum())

transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              139
dtype: int64


Pembersihan dari data duplicate

In [57]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 15
transaction_id duplicate: 15
Jumlah baris setelah drop_duplicates(): 500


Standarisasi data

Standarisasi teks kategorikal

In [58]:
for col in["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

print(df["category"].unique())
print(df["payment_method"].unique())

<StringArray>
['Olahraga', 'Elektronik', 'Buku', 'Kesehatan', 'Fashion', 'Rumah Tangga']
Length: 6, dtype: string
<StringArray>
['COD', 'Transef Bank', 'E-Wallet', 'Kartu Kredit', <NA>]
Length: 5, dtype: string


Standarisasi penulisan harga

In [59]:
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".","").replace(",",".")
  try:
    return float(x)
  except ValueError:
    return np.nan
df["price"] = df["price"].apply(bersihkan_harga)

Standarisasi penulisan tanggal

In [60]:
def parse_tanggal(x):
  for fmt in ("%Y-%m-%d", "%d%m%Y", "%d-%m-%Y"):
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

Finalisasi bentuk data

In [61]:
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

Ekspor data bersih

In [62]:
df_bersih = df.copy()
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 500 baris


Menyimpan data

In [68]:
output_file_path = os.path.join(DIR_SIMPAN, "transaksi_mentah_tugas.csv")
df.to_csv(output_file_path, index=False)
print("Dataset bersih tersimpan:", len(df), "baris")
print(f"File saved to: {output_file_path}")

Dataset bersih tersimpan: 500 baris
File saved to: /content/drive/MyDrive/BigData/TugasP2/transaksi_mentah_tugas.csv


In [69]:
output_file_path = os.path.join(DIR_SIMPAN, "transaksi_bersih_tugas.csv")
df.to_csv(output_file_path, index=False)
print("Dataset bersih tersimpan:", len(df), "baris")
print(f"File saved to: {output_file_path}")

Dataset bersih tersimpan: 500 baris
File saved to: /content/drive/MyDrive/BigData/TugasP2/transaksi_bersih_tugas.csv


Memeriksa harga tidak valid

In [64]:
df["is_valid_price"] = df["price"] > 0

jumlah_invalid = (~df["is_valid_price"]).sum()
print(f"\nJumlah transaksi dengan harga tidak valid (<= 0 atau NaN): {jumlah_invalid}")

if jumlah_invalid > 0:
    print("--- Sampel Data Harga Tidak Valid ---")
    display(df[~df["is_valid_price"]][["transaction_id", "product_name", "price", "is_valid_price"]].head())


file_path = f"{DIR_SIMPAN}/transaksi_bersih.csv"
df.to_csv(file_path, index=False)
print(f"\nDataset bersih (dengan is_valid_price) berhasil disimpan di: {file_path}")

print(df[["transaction_id", "product_name", "price", "is_valid_price"]])


Jumlah transaksi dengan harga tidak valid (<= 0 atau NaN): 0

Dataset bersih (dengan is_valid_price) berhasil disimpan di: /content/drive/MyDrive/BigData/TugasP2/transaksi_bersih.csv
    transaction_id     product_name      price  is_valid_price
0         TRX00490       Aspernatur    25000.0            True
1         TRX00430  Voluptates Lite    50000.0            True
2         TRX00083     Deserunt Pro    50000.0            True
3         TRX00121       Quia Basic   250000.0            True
4         TRX00222     Magnam Basic    75000.0            True
..             ...              ...        ...             ...
509       TRX00408         In Basic  5000000.0            True
510       TRX00212       Enim Basic  1200000.0            True
511       TRX00068        Illum Max    50000.0            True
513       TRX00197               Ex   250000.0            True
514       TRX00176     Pariatur Max    75000.0            True

[500 rows x 4 columns]


Hitung jumlah transaksi per category

In [65]:

jumlah_per_kategori = df["category"].value_counts()

print("--- Jumlah Transaksi per Kategori ---")
print(jumlah_per_kategori)

--- Jumlah Transaksi per Kategori ---
category
Fashion         107
Elektronik       86
Kesehatan        82
Olahraga         80
Buku             75
Rumah Tangga     70
Name: count, dtype: Int64
